# Merge CRISPR and RNA-seq Data

This notebook integrates CRISPR sgRNA screening results with RNA-seq DESeq2 differential expression results to generate the input file needed for quadrant analysis, and labels each gene with its differential expression status based on RNA-seq results.

## Input Files
- CRISPR: `/path/to/CRISPR/results/sgRNA_info_with_category_zlu_lib.csv`
- RNA-seq: `/path/to/project/00.Basicanalysis/RNA-seq/deg/deseq2_*.txt` (three files)

## Output Format
CSV file containing the following columns:
- `gene_id`: Ensembl gene ID
- `gene_symbol`: Gene symbol
- `crispr_score`: CRISPR gene-level log2FC (median across multiple sgRNAs)
- `crispr_category`: CRISPR effect classification (Depleted/Slightly Enriched/Moderately Enriched/Highly Enriched)
- `rna_log2fc`: RNA-seq log2 Fold Change
- `rna_padj`: RNA-seq adjusted p-value
- `rna_deg_status`: RNA-seq differential expression status (Up/Down/No_change)


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob

print('✓ Libraries imported successfully')


In [ ]:
# === Parameter Settings ===
CRISPR_SGRNA_CSV = '/path/to/CRISPR/results/sgRNA_info_with_category_zlu_lib.csv'
DESEQ_DIR = '/path/to/project/00.Basicanalysis/RNA-seq/deg'
DESEQ_PATTERN = f'{DESEQ_DIR}/deseq2_*.txt'
OUTPUT_CSV = '/path/to/CRISPR/results/crispr_rnaseq_merged.csv'

# Differential expression thresholds
RNA_LOG2FC_CUTOFF = 0.0   # Only determine direction; change to 1.0 if |log2FC|>1 is required
RNA_PADJ_CUTOFF = 0.05

print('Input files:')
print('  CRISPR CSV:', CRISPR_SGRNA_CSV)
print('  DESeq2 file pattern:', DESEQ_PATTERN)
print('Output file:', OUTPUT_CSV)


In [ ]:
# === Step 1: Process CRISPR Data ===
print('=== Processing CRISPR Data ===')

# Read CRISPR data
cr_df = pd.read_csv(CRISPR_SGRNA_CSV)
print(f'Read {len(cr_df)} sgRNA records')

# Extract gene ID from sgRNA name (e.g., ENSSSCG00000001849_... -> ENSSSCG00000001849)
cr_df['gene_id'] = cr_df['sgRNA'].astype(str).str.split('_').str[0]

# Use GeneID_y as gene symbol
if 'GeneID_y' in cr_df.columns:
    cr_df['gene_symbol'] = cr_df['GeneID_y'].astype(str)
else:
    cr_df['gene_symbol'] = cr_df['GeneID_x'].astype(str)

# Check required columns
if 'log2(Fold_change)' not in cr_df.columns:
    raise ValueError('CRISPR file missing log2(Fold_change) column')
if 'category' not in cr_df.columns:
    raise ValueError('CRISPR file missing category column')

# Aggregate to gene level: take median log2FC across multiple sgRNAs per gene, and keep category
def aggregate_gene(group):
    return pd.Series({
        'crispr_score': group['log2(Fold_change)'].median(),
        'crispr_category': group['category'].mode()[0] if len(group['category'].mode()) > 0 else group['category'].iloc[0]
    })

crispr_gene = (
    cr_df.groupby(['gene_id', 'gene_symbol'])
        .apply(aggregate_gene)
        .reset_index()
)

print(f'Aggregated to {len(crispr_gene)} genes')
print('\nCRISPR gene data preview:')
display(crispr_gene.head())
print('\n✓ CRISPR data processing complete')


In [ ]:
# === Step 2: Process RNA-seq Data ===
print('=== Processing RNA-seq Data ===')

def read_deseq_file(fpath: str) -> pd.DataFrame:
    """
    Read DESeq2 output file.
    Assumes the first column (unnamed) is the gene ID.
    """
    try:
        # Try reading as TAB-separated, with the first column as index
        df = pd.read_csv(fpath, sep='\t', index_col=0)
    except Exception:
        # Fall back to default separator
        df = pd.read_csv(fpath, index_col=0)
    
    # Convert index to column
    df.reset_index(inplace=True)
    df.rename(columns={df.columns[0]: 'gene_id'}, inplace=True)
    
    # Clean column names
    df.columns = [str(c).strip() for c in df.columns]
    
    # Identify log2FC and padj columns (case-insensitive)
    cols_lower = {c.lower(): c for c in df.columns}
    
    def find_col(names):
        for n in names:
            if n in cols_lower:
                return cols_lower[n]
        return None
    
    col_log2fc = find_col(['log2foldchange', 'log2fc', 'logfc'])
    col_padj = find_col(['padj', 'qval', 'fdr', 'adj_pval'])
    
    if col_log2fc is None:
        print(f'  Warning: {Path(fpath).name} missing log2FC column')
        return pd.DataFrame(columns=['gene_id', 'rna_log2fc', 'rna_padj'])
    
    if col_padj is None:
        col_padj = 'padj_placeholder'
        df[col_padj] = np.nan
    
    # Build output dataframe
    out = pd.DataFrame({
        'gene_id': df['gene_id'].astype(str),
        'rna_log2fc': pd.to_numeric(df[col_log2fc], errors='coerce'),
        'rna_padj': pd.to_numeric(df[col_padj], errors='coerce')
    })
    
    return out

# Read all DESeq2 files
files = sorted(glob(DESEQ_PATTERN))
print(f'Found {len(files)} DESeq2 files')

if len(files) == 0:
    raise FileNotFoundError(f'No matching files found: {DESEQ_PATTERN}')

rna_list = []
for f in files:
    print(f'  Reading: {Path(f).name}')
    res = read_deseq_file(f)
    if not res.empty:
        rna_list.append(res)
        print(f'    -> {len(res)} genes')

if not rna_list:
    raise ValueError('All RNA-seq files failed to load')

# Merge all files
rna_all = pd.concat(rna_list, ignore_index=True)
print(f'\nTotal records after merging: {len(rna_all)}')

# Aggregate: for the same gene, select the record with the smallest padj
def pick_best_record(group):
    """
    For multiple records of the same gene, select the most significant one (smallest padj).
    If all padj values are NaN, select the one with the largest |log2FC|.
    """
    # Sort by padj ascending
    sorted_group = group.sort_values('rna_padj', na_position='last')
    best = sorted_group.iloc[0]
    
    # If the best record's padj is NaN, select by |log2FC|
    if pd.isna(best['rna_padj']):
        sorted_by_fc = group.reindex(
            group['rna_log2fc'].abs().sort_values(ascending=False).index
        )
        best = sorted_by_fc.iloc[0]
    
    # Note: gene_id is already the groupby key, no need to duplicate in return
    return pd.Series({
        'rna_log2fc': best['rna_log2fc'],
        'rna_padj': best['rna_padj']
    })

rna_gene = rna_all.groupby('gene_id', dropna=True).apply(pick_best_record, include_groups=False).reset_index()
print(f'Aggregated to {len(rna_gene)} genes')
print('\nRNA-seq gene data preview:')
display(rna_gene.head())
print('\n✓ RNA-seq data processing complete')


In [ ]:
# === Step 3: Merge CRISPR and RNA-seq Data ===
print('=== Merging Data ===')

# Left join based on gene_id
merged = crispr_gene.merge(
    rna_gene[['gene_id', 'rna_log2fc', 'rna_padj']],
    on='gene_id',
    how='left'
)

# Match statistics
total = len(merged)
matched = merged['rna_log2fc'].notna().sum()
missing = total - matched

print(f'Total CRISPR genes: {total}')
print(f'Successfully matched with RNA-seq: {matched} ({matched/total*100:.1f}%)')
print(f'Unmatched: {missing} ({missing/total*100:.1f}%)')

# Build final output (including crispr_category)
final = merged[['gene_id', 'gene_symbol', 'crispr_score', 'crispr_category', 'rna_log2fc', 'rna_padj']].copy()

# Label up-regulated/down-regulated/no change based on RNA-seq results (using English)
def classify_deg(row):
    if pd.isna(row['rna_log2fc']) or pd.isna(row['rna_padj']):
        return 'No_change'
    if row['rna_padj'] < RNA_PADJ_CUTOFF:
        if row['rna_log2fc'] > RNA_LOG2FC_CUTOFF:
            return 'Up'
        elif row['rna_log2fc'] < -RNA_LOG2FC_CUTOFF:
            return 'Down'
    return 'No_change'

final['rna_deg_status'] = final.apply(classify_deg, axis=1)

print('\nMerged data preview:')
display(final.head(10))
print('\n✓ Data merge complete')


In [ ]:
# === Step 4: Save Results ===
print('=== Saving Results ===')

# Ensure output directory exists
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)

# Save as CSV
final.to_csv(OUTPUT_CSV, index=False)

print(f'✓ Results saved: {OUTPUT_CSV}')
print(f'  Total rows: {len(final)}')
print(f'  File size: {round(Path(OUTPUT_CSV).stat().st_size / 1024, 2)} KB')

# Display statistics
print('\nData statistics:')
print(f'  CRISPR score range: {final["crispr_score"].min():.2f} ~ {final["crispr_score"].max():.2f}')
print(f'\nCRISPR category distribution:')
print(final['crispr_category'].value_counts().sort_index())

if matched > 0:
    print(f'\nRNA-seq data statistics:')
    print(f'  log2FC range: {final["rna_log2fc"].min():.2f} ~ {final["rna_log2fc"].max():.2f}')
    print(f'  padj < 0.05: {(final["rna_padj"] < 0.05).sum()} genes')
    print(f'\nRNA-seq differential expression distribution:')
    print(f'  Up: {(final["rna_deg_status"] == "Up").sum()}')
    print(f'  Down: {(final["rna_deg_status"] == "Down").sum()}')
    print(f'  No_change: {(final["rna_deg_status"] == "No_change").sum()}')

print('\n✓ All complete!')
print(f'\nNext step: Use {OUTPUT_CSV} as input for Quadrant_Classification.ipynb')


In [ ]:
# === Debug: Check final DataFrame contents ===
print('=== Debug Information ===')
print(f'final DataFrame columns: {list(final.columns)}')
print(f'final DataFrame shape: {final.shape}')
print(f'\nrna_deg_status column unique values:')
print(final['rna_deg_status'].value_counts())
print(f'\nrna_deg_status column data type: {final["rna_deg_status"].dtype}')
print(f'\nFirst 10 rows:')
display(final.head(10))
